### Lab Assignment: Commercial Data Analysis

### University of Virginia
### DS 7200: Distributed Computing
### Last Updated: August 20, 2023

---

Sabine Segaloff

bhj3vc

### INSTRUCTIONS  
In this assignment, you will work with a dataset containing information about businesses.  
Each record is a business location.  Follow the steps below, writing and running the code in blocks, and displaying the solutions.  

Each question part is worth 1 POINT, for a total of 15 POINTS.

Hint: reaching deeper fields in json hierarchy can be done like this:  

`df.select('address.street_number')`

---

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
        .appName("comm") \
        .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/18 19:59:34 WARN Utils: Your hostname, Beans-Legion, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/09/18 19:59:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/sabine/projects/distributed_computing/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/18 19:59:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
# note that read.json can read a zipped JSON directly

**1. (1 PT) Read in the dataset and show the number of records**

In [3]:
# x = spark.read.json('/standard/ds7200-apt4c/large_datasets/part-00000-a159c41a-bc58-4476-9b78-c437667f9c2b-c000.json.gz')
x = spark.read.json('part-00000-a159c41a-bc58-4476-9b78-c437667f9c2b-c000.json.gz')

**2. (1 PT) Show the first 5 records**

In [4]:
x.show(5)

+--------------------+--------------------+--------------------+----------------+----+--------------------+--------------------+--------------------+
|             address|       business_tags|               hours|              id|menu|             reviews|                urls|             webpage|
+--------------------+--------------------+--------------------+----------------+----+--------------------+--------------------+--------------------+
|{Woodburn, {45.15...|                NULL|                NULL|000023995a540868|NULL|                  []|{woodburn.k12.or....|{Educational Tech...|
|{Hialeah, {25.884...|{[], [{has_atm, Y...|{NULL, 1900, NULL...|0000821a1394916e|NULL|                NULL|{NULL, [yelp.com]...|                NULL|
|{Rochester, {43.1...|{[], [{accepts_cr...|{NULL, 1700, NULL...|000136e65d50c3b7|NULL|[{New (to me) qui...|{usps.com, [yelp....|{Welcome | USPS G...|
|{West Palm Beach,...|                NULL|                NULL|00014329a70b9869|NULL|              

**3. (1 PT) Show the first 5 street addresses which are not null**  

In [5]:
# print DF structure as a readable tree (so that I can find the field name for the street addresses under the address struct)
x.select('address').printSchema(3)

root
 |-- address: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- coordinates: struct (nullable = true)
 |    |    |-- lat: double (nullable = true)
 |    |    |-- lon: double (nullable = true)
 |    |-- country: string (nullable = true)
 |    |-- county: string (nullable = true)
 |    |-- full_address: string (nullable = true)
 |    |-- highway_number: string (nullable = true)
 |    |-- is_headquarters: boolean (nullable = true)
 |    |-- is_parsed: boolean (nullable = true)
 |    |-- post_direction: string (nullable = true)
 |    |-- pre_direction: string (nullable = true)
 |    |-- secondary_number: string (nullable = true)
 |    |-- state: string (nullable = true)
 |    |-- street: string (nullable = true)
 |    |-- street_address: string (nullable = true)
 |    |-- street_number: string (nullable = true)
 |    |-- street_type: string (nullable = true)
 |    |-- type_of_address: string (nullable = true)
 |    |-- zip: string (nullable = true)
 |    |-- 

In [6]:
# since we only want the street addresses, we should use `street_address` rather than `full_address`?
x.select('address.street_address').show(10)

x.select('address.full_address').show(10)

x.select('address.street_address').show(10)

+------------------+
|    street_address|
+------------------+
|              NULL|
|              NULL|
|              NULL|
|              NULL|
|              NULL|
|              NULL|
|              NULL|
|              NULL|
|              NULL|
|Cooper Contracting|
+------------------+
only showing top 10 rows
+------------+
|full_address|
+------------+
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
|        NULL|
+------------+
only showing top 10 rows
+------------------+
|    street_address|
+------------------+
|              NULL|
|              NULL|
|              NULL|
|              NULL|
|              NULL|
|              NULL|
|              NULL|
|              NULL|
|              NULL|
|Cooper Contracting|
+------------------+
only showing top 10 rows


*So street address and full address appear not helpful in most cases. So I'm going to try to create the street address by grabbing the street number and street and street type*

In [7]:
x.select('address.street_number', 'address.street', 'address.street_type').show(10)

+-------------+--------------------+-----------+
|street_number|              street|street_type|
+-------------+--------------------+-----------+
|          965|        Boones Ferry|         Rd|
|         1137|                68th|         St|
|         1614|            Penfield|         Rd|
|          846|                Park|         Pl|
|          403|                Main|         St|
|         2045|Brookwood Medical...|         Dr|
|         3320|               Hwy 6|        Hwy|
|       13237b|                41st|         Rd|
|         7091|                Main|         St|
|         NULL|                NULL|       NULL|
+-------------+--------------------+-----------+
only showing top 10 rows


In [8]:
# to select from a nested field to call is.NotNull(), I need to use col() function
from pyspark.sql.functions import col

print('First 5 non-null street addresses from street_number, street, street_type:')
x.select('address.street_number', 'address.street', 'address.street_type').filter(col('address.street').isNotNull()).show(5)

# check street_address for curiosity
print("\nJust out of curiosity I'm checking street_address field too:")
x.select('address.street_address').filter(col('address.street_address').isNotNull()).show(5)

First 5 non-null street addresses from street_number, street, street_type:
+-------------+------------+-----------+
|street_number|      street|street_type|
+-------------+------------+-----------+
|          965|Boones Ferry|         Rd|
|         1137|        68th|         St|
|         1614|    Penfield|         Rd|
|          846|        Park|         Pl|
|          403|        Main|         St|
+-------------+------------+-----------+
only showing top 5 rows

Just out of curiosity I'm checking street_address field too:
+-------------------+
|     street_address|
+-------------------+
| Cooper Contracting|
|          Route 607|
|Bush St & Kearny St|
|          S 14th St|
|             Rr 474|
+-------------------+
only showing top 5 rows


**ANSWER for 3 directly above this cell**

**4. (1 PT) Location**  

Count the number of records where the city is Phoenix

In [9]:
x.filter(col('address.city') == 'Phoenix').count()

26/09/18 19:59:43 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


762

**ANSWER for 4 directly above this cell**

**5. (1 PT) Hours**  

Count the number of records where closing time on Thursday is 8pm

In [10]:
x.select('hours').printSchema(2)

root
 |-- hours: struct (nullable = true)
 |    |-- any_day_is_24: boolean (nullable = true)
 |    |-- friday_close: string (nullable = true)
 |    |-- friday_lb: long (nullable = true)
 |    |-- friday_open: string (nullable = true)
 |    |-- friday_total_seconds: long (nullable = true)
 |    |-- hours: struct (nullable = true)
 |    |-- monday_close: string (nullable = true)
 |    |-- monday_lb: long (nullable = true)
 |    |-- monday_open: string (nullable = true)
 |    |-- monday_total_seconds: long (nullable = true)
 |    |-- saturday_close: string (nullable = true)
 |    |-- saturday_lb: long (nullable = true)
 |    |-- saturday_open: string (nullable = true)
 |    |-- saturday_total_seconds: long (nullable = true)
 |    |-- sunday_close: string (nullable = true)
 |    |-- sunday_lb: long (nullable = true)
 |    |-- sunday_open: string (nullable = true)
 |    |-- sunday_total_seconds: long (nullable = true)
 |    |-- thursday_close: string (nullable = true)
 |    |-- thursday_lb:

In [11]:
x.select('hours.thursday_close').show(5)

+--------------+
|thursday_close|
+--------------+
|          NULL|
|          1800|
|          1700|
|          NULL|
|          1700|
+--------------+
only showing top 5 rows


In [12]:
# check if 8pm might be stored in ways other than 2000
from pyspark.sql.functions import length
x.select(length(col('hours.thursday_close'))).distinct().show(5)

+----------------------------+
|length(hours.thursday_close)|
+----------------------------+
|                           6|
|                           4|
|                        NULL|
+----------------------------+



*So not all the times are HHMM. Now I should see what the 6 character values are*

In [13]:
x.select('hours.thursday_close').filter(length(col('hours.thursday_close')) == 6).show(5)

+--------------+
|thursday_close|
+--------------+
|        000000|
|        000000|
|        003000|
|        000000|
|        000000|
+--------------+
only showing top 5 rows


*HHMMSS*

In [14]:
# x.filter((col('hours.thursday_close') == '2000') | (col('hours.thursday_close') == '200000')).count()

# this way seems neater since i'm checking multiple exact matches against one column. but the above also worked
x.filter(col('hours.thursday_close').isin(['2000', '200000'])).count()


3313

**ANSWER for 5 directly above this cell**

**6. (1 PT) Location and Hours**  

Count the number of records where city is Phoenix and closing time on Thursday is 8pm

In [15]:
x.filter((col('address.city') == 'Phoenix') & (col('hours.thursday_close').isin(['2000', '200000']))).count()

12

**ANSWER for 6 directly above this cell**

**7. (1 PT) Price Range**  

Price range is quoted in number of dollar signs.  Count the number of records with price range greater than or equal to two.

In [16]:
# check out schema (I guess that price is in menu)
x.select('menu').printSchema(2)

root
 |-- menu: struct (nullable = true)
 |    |-- price_range: string (nullable = true)
 |    |-- url: string (nullable = true)



In [17]:
x.select(col('menu.price_range')).distinct().show(10)

+-----------+
|price_range|
+-----------+
|          3|
|          1|
|          4|
|          2|
|       NULL|
+-----------+



*So they are stored as single digit numbers as strings that refer to number of $, not as \$'s themselves*

In [18]:
# x.filter((col('menu.price_range')).isin(['2','3','4'])).count()

# the above works, but to make this more flexible rather than hardcoding, I will just cast the column to an int data type and use math
# the .cast() method works on a Column object, not a dataframe
x.select(col('menu.price_range')).filter(col('menu.price_range').cast('int') >=2).distinct().show(100)

+-----------+
|price_range|
+-----------+
|          3|
|          4|
|          2|
+-----------+



*okay so the nulls don't cause an issue*

In [19]:
x.filter(col('menu.price_range').cast('int') >=2).count()

1135

**ANSWER for 7 directly above this cell**

**8. (1 PT) COMPANY HEADQUARTERS**  

For the `address.is_headquarters` field:  
how many locations are HQ / are NOT HQ / are null?

In [20]:
x.select('address').printSchema(2)

root
 |-- address: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- coordinates: struct (nullable = true)
 |    |-- country: string (nullable = true)
 |    |-- county: string (nullable = true)
 |    |-- full_address: string (nullable = true)
 |    |-- highway_number: string (nullable = true)
 |    |-- is_headquarters: boolean (nullable = true)
 |    |-- is_parsed: boolean (nullable = true)
 |    |-- post_direction: string (nullable = true)
 |    |-- pre_direction: string (nullable = true)
 |    |-- secondary_number: string (nullable = true)
 |    |-- state: string (nullable = true)
 |    |-- street: string (nullable = true)
 |    |-- street_address: string (nullable = true)
 |    |-- street_number: string (nullable = true)
 |    |-- street_type: string (nullable = true)
 |    |-- type_of_address: string (nullable = true)
 |    |-- zip: string (nullable = true)
 |    |-- zip_suffix: string (nullable = true)



*is_headquarters is type boolean, nullable, so possible states are `True`, `False`, and `NULL`.*

In [21]:
# I could count each separately, but groupBy would make more sense
x.groupBy('address.is_headquarters').count().show()

+---------------+-----+
|is_headquarters|count|
+---------------+-----+
|           NULL|87625|
|           true|  318|
|          false|66736|
+---------------+-----+



**ANSWER for 8 directly above this cell**

**9. (1 PT) Webpage URLs**  

Register the dataframe as a temp table.  
Next, use Spark SQL to select only the webpage title column, filtering on rows where the webpage url (accessed under `webpage.url`) is *Target.com*. 

Show only one resulting row and don't truncate the output.

In [22]:
x.select('webpage').printSchema(3)

root
 |-- webpage: struct (nullable = true)
 |    |-- content: string (nullable = true)
 |    |-- count: long (nullable = true)
 |    |-- elapsed: double (nullable = true)
 |    |-- success: boolean (nullable = true)
 |    |-- timestamp: string (nullable = true)
 |    |-- title: string (nullable = true)
 |    |-- url: string (nullable = true)
 |    |-- urlhash: string (nullable = true)
 |    |-- validurl: string (nullable = true)



In [23]:
# registering as a temp table is a DataFrame method that takes a name string
# registerTempTable(name) is an older, deprecated method according to my research, so I will use createOrReplaceTempView(name)
    # "Replace" means that if you run the cell twice (or rerun the notebook), it won't error out and complain it already exists, it just overwrites instead

x.createOrReplaceTempView('businesses')

In [24]:
# now I use spark.sql(query_string) and pass a string with an SQL query and it will return a dataframe
query = """
SELECT webpage.title
FROM businesses
WHERE webpage.url = 'Target.com'
"""

spark.sql(query).show(n=1, truncate=False)

+-------------------------------+
|title                          |
+-------------------------------+
|Target : Expect More. Pay Less.|
+-------------------------------+
only showing top 1 row


**ANSWER for 9 directly above this cell**

**10. (1 PT) Analysis on Ratings**  

The reviews contains information such as the number of stars for each review (the *rating*).  
The ratings are stored in an array (`reviews.stars`) for each business location (you should check for yourself). Return the top five most common rating arrays.  For example, an array might look like: 
[5, 5]



In [25]:
x.select('reviews').printSchema(5)

root
 |-- reviews: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- content: string (nullable = true)
 |    |    |-- date: string (nullable = true)
 |    |    |-- dislikes: long (nullable = true)
 |    |    |-- gender: string (nullable = true)
 |    |    |-- id: string (nullable = true)
 |    |    |-- language: string (nullable = true)
 |    |    |-- likes: long (nullable = true)
 |    |    |-- source: string (nullable = true)
 |    |    |-- stars: long (nullable = true)
 |    |    |-- tags: array (nullable = true)
 |    |    |    |-- element: string (containsNull = true)
 |    |    |-- url: string (nullable = true)
 |    |    |-- user: string (nullable = true)
 |    |    |-- user_id: string (nullable = true)



*so reviews is an array and each item in the array (each element) is a struct with the fields content, date, dislikes, ..., etc.*

*since reviews is an array of structs, and stars is a field inside that struct, col('reviews.stars') will pull stars out of every element in the array and return an array of longs for each row (one star value per review that business has), not a single value.*

In [26]:
# if I group by reviews.stars, I'll get a single group for each unique rating combination, then I will use count() to see how many are in the groups
# then I can sort by count, descending and display the first 5 [can also use orderBy(), they are aliases similar to .filter()/.where()]
x.groupBy('reviews.stars').count().sort('count', ascending = False).show(5)

+------+-----+
| stars|count|
+------+-----+
|  NULL|74679|
|    []|42419|
|   [5]| 4258|
|[NULL]| 3067|
|[5, 5]| 1610|
+------+-----+
only showing top 5 rows


**ANSWER for 10 directly above this cell**

**11. More work with Ratings**  

For this question, you will filter out null ratings and then compute the average rating for each business location (using the field: `id`).


a) (1 PT) Create a new dataframe retaining two fields: `id`, `reviews.stars`


In [27]:
id_ratings = x.select('id','reviews.stars')

id_ratings.show(5)

+----------------+------+
|              id| stars|
+----------------+------+
|000023995a540868|    []|
|0000821a1394916e|  NULL|
|000136e65d50c3b7|[4, 4]|
|00014329a70b9869|  NULL|
|00031c0a83f00657|  NULL|
+----------------+------+
only showing top 5 rows


b) (1 PT) Create a row for each rating  
hint: use the `withColumn()` and `explode()` functions  
you will need to import the `explode()` function by issuing:

`from pyspark.sql.functions import explode`


In [28]:
# .withColumn(new_col_name, expression) is a method that adds a new column or overwrites an existing one (if you reuse same name)
# it takes two arguments, the name of column you are creating/replacing, and the expression that computes it

# each row in id_ratings has one id paired with an array of stars. 'explode' means spark will take that array and flatten it into
# multiple rows, one row per array element, duplicating the id for each. however, explode produces a column like expression representing one row per array element
# but doesn't put it anywhere, which is what .withColumn() is for

from pyspark.sql.functions import explode

id_ratings = id_ratings.withColumn('stars', explode('stars'))

id_ratings.show(5)

+----------------+-----+
|              id|stars|
+----------------+-----+
|000136e65d50c3b7|    4|
|000136e65d50c3b7|    4|
|0003b7589a4e12a0|    5|
|00045f958e4bb02a| NULL|
|00045f958e4bb02a| NULL|
+----------------+-----+
only showing top 5 rows


c) (1 PT) Return a count of the number of ratings in this dataframe

In [29]:
id_ratings.count()

600082

d) (1 PT) Drop rows where the rating is null, and return a count of the number of non-null ratings

In [30]:
# confirm no weird nested nulls or something
id_ratings.select('stars').distinct().show()

+-----+
|stars|
+-----+
|    5|
|    1|
|    3|
|    2|
|    4|
| NULL|
+-----+



In [31]:
# id_ratings = id_ratings.filter(col('stars').isNotNull())

# trying another way even though above works great. .dropna() will check all columns or a specified subset of columns (pass as list)
id_ratings = id_ratings.dropna(subset=['stars'])

id_ratings.show(5)

+----------------+-----+
|              id|stars|
+----------------+-----+
|000136e65d50c3b7|    4|
|000136e65d50c3b7|    4|
|0003b7589a4e12a0|    5|
|00059519f0dba1b4|    1|
|00059519f0dba1b4|    5|
+----------------+-----+
only showing top 5 rows


In [32]:
id_ratings.count()

538241

**ANSWER for 11d directly above this cell**

e) (1 PT) Compute the average rating, grouped by `id`. After the average is computed, sort by `id` in ascending order and show the top 10 records.  
 
hint:   
this can all be done in one line using the `agg()` function  
this `id` should be at the top: 000136e65d50c3b7

In [33]:
from pyspark.sql.functions import avg # or mean

id_ratings.groupBy('id').agg(avg('stars')).sort('id').show(10)

+----------------+------------------+
|              id|        avg(stars)|
+----------------+------------------+
|000136e65d50c3b7|               4.0|
|0003b7589a4e12a0|               5.0|
|00059519f0dba1b4|3.3333333333333335|
|000a1df4c8e0ecd2|               4.6|
|000c7b7a30623083|               5.0|
|000c9ffc8b89af03|               3.0|
|000de20baa847ecc|1.6666666666666667|
|001064359d9f162f|               5.0|
|0010c9f495d87dd7|               3.0|
|0017774db5e6400a| 4.333333333333333|
+----------------+------------------+
only showing top 10 rows
